In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')

np.seterr(all='warn', over='ignore')
pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from advanced_evaluation.advanced_evaluation import SampleOutcomesAdvanced, SampleOutcomesAdvancedPCR

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
model_name = 'pcr'

n_processes = 128
batch_size = 8

log_name = 'test'

with open('../transformed_event_logs/PCR_start_end_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)

#test_event_log['time:timestamp'] = test_event_log['time:timestamp_complete']
test_event_log['case:concept:name'] = test_event_log['case:concept:name'].astype(str)
known_activities = ['Callback timeout', 'Export result', 'Export to EMS', 'Match patient data', 'Receive sample state', 'Send notification', 'Wait for plate validation', 'timeout']

ii1 = ['intercase_n_1__Callback timeout', 'intercase_n_1__Export result', 'intercase_n_1__Export to EMS', 'intercase_n_1__Match patient data', 'intercase_n_1__Receive sample state', 'intercase_n_1__Send notification', 'intercase_n_1__Wait for plate validation', 'intercase_n_1__timeout']
ii3 = ['intercase_n_3__Export result_Callback timeout_Send notification', 'intercase_n_3__Export result_Export to EMS_Callback timeout', 'intercase_n_3__Export result_Export to EMS_Send notification', 'intercase_n_3__Export result_Send notification_Callback timeout', 'intercase_n_3__Export to EMS_Callback timeout_Send notification', 'intercase_n_3__Export to EMS_Export result_Callback timeout', 'intercase_n_3__Export to EMS_Export result_Send notification', 'intercase_n_3__Export to EMS_Send notification_Callback timeout', 'intercase_n_3__Match patient data', 'intercase_n_3__Match patient data_Match patient data', 'intercase_n_3__Match patient data_Match patient data_Match patient data', 'intercase_n_3__Match patient data_Match patient data_Receive sample state', 'intercase_n_3__Match patient data_Match patient data_Send notification', 'intercase_n_3__Match patient data_Receive sample state_Callback timeout', 'intercase_n_3__Match patient data_Receive sample state_Export result', 'intercase_n_3__Match patient data_Receive sample state_Export to EMS', 'intercase_n_3__Match patient data_Receive sample state_Send notification', 'intercase_n_3__Match patient data_Send notification_Receive sample state', 'intercase_n_3__Match patient data_Wait for plate validation', 'intercase_n_3__Match patient data_Wait for plate validation_Receive sample state', 'intercase_n_3__Match patient data_Wait for plate validation_Send notification', 'intercase_n_3__Match patient data_Wait for plate validation_timeout', 'intercase_n_3__Match patient data_timeout', 'intercase_n_3__Match patient data_timeout_Match patient data', 'intercase_n_3__Match patient data_timeout_Receive sample state', 'intercase_n_3__Match patient data_timeout_Send notification', 'intercase_n_3__Match patient data_timeout_Wait for plate validation', 'intercase_n_3__Receive sample state_Callback timeout_Send notification', 'intercase_n_3__Receive sample state_Export result_Export to EMS', 'intercase_n_3__Receive sample state_Export result_Send notification', 'intercase_n_3__Receive sample state_Export to EMS_Export result', 'intercase_n_3__Receive sample state_Export to EMS_Send notification', 'intercase_n_3__Receive sample state_Send notification_Callback timeout', 'intercase_n_3__Receive sample state_Send notification_Export result', 'intercase_n_3__Receive sample state_Send notification_Export to EMS', 'intercase_n_3__Send notification_Export result_Export to EMS', 'intercase_n_3__Send notification_Export to EMS_Export result', 'intercase_n_3__Send notification_Receive sample state_Callback timeout', 'intercase_n_3__Send notification_Receive sample state_Export result', 'intercase_n_3__Send notification_Receive sample state_Export to EMS', 'intercase_n_3__Wait for plate validation', 'intercase_n_3__Wait for plate validation_Match patient data', 'intercase_n_3__Wait for plate validation_Match patient data_Receive sample state', 'intercase_n_3__Wait for plate validation_Match patient data_Send notification', 'intercase_n_3__Wait for plate validation_Match patient data_timeout', 'intercase_n_3__Wait for plate validation_Receive sample state', 'intercase_n_3__Wait for plate validation_Receive sample state_Callback timeout', 'intercase_n_3__Wait for plate validation_Receive sample state_Export result', 'intercase_n_3__Wait for plate validation_Receive sample state_Export to EMS', 'intercase_n_3__Wait for plate validation_Receive sample state_Send notification', 'intercase_n_3__Wait for plate validation_Send notification_Receive sample state', 'intercase_n_3__Wait for plate validation_timeout', 'intercase_n_3__Wait for plate validation_timeout_Match patient data', 'intercase_n_3__Wait for plate validation_timeout_Receive sample state', 'intercase_n_3__Wait for plate validation_timeout_Send notification', 'intercase_n_3__timeout', 'intercase_n_3__timeout_Match patient data', 'intercase_n_3__timeout_Match patient data_Match patient data', 'intercase_n_3__timeout_Match patient data_Receive sample state', 'intercase_n_3__timeout_Match patient data_Send notification', 'intercase_n_3__timeout_Match patient data_Wait for plate validation', 'intercase_n_3__timeout_Receive sample state_Callback timeout', 'intercase_n_3__timeout_Receive sample state_Export result', 'intercase_n_3__timeout_Receive sample state_Export to EMS', 'intercase_n_3__timeout_Receive sample state_Send notification', 'intercase_n_3__timeout_Send notification_Receive sample state', 'intercase_n_3__timeout_Wait for plate validation', 'intercase_n_3__timeout_Wait for plate validation_Match patient data', 'intercase_n_3__timeout_Wait for plate validation_Receive sample state', 'intercase_n_3__timeout_Wait for plate validation_Send notification']

In [3]:
drbart_model_path = '../../../models/advanced/'+model_name+'/csdac_ii1/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvancedPCR, {
                                                        'activity_key' : 'concept:name',
                                                        'timestamp_key' : 'time:timestamp_start',
                                                        'categorical_args' : ['concept_name',
                                                                              'day_of_week',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda inter_instance_counts, inter_instance_column_names : [0 if inter_instance_column_name not in inter_instance_counts else inter_instance_counts[inter_instance_column_name] for inter_instance_column_name in inter_instance_column_names])(inter_instance_counts, self.inter_instance_column_names)'
                                                                             ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'inter_instance_column_names' : ii1,
                                                        'strict_parsing' : False,
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, multiprocessing=True)

100%|██████████| 1202/1202 [00:06<00:00, 177.19it/s]


In [4]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [5]:
np.mean(get_pscores(likelihoods_A))

np.float64(18493.639254846472)

In [6]:
drbart_model_path = '../../../models/advanced/'+model_name+'/csdac_ii3/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvancedPCR, {
                                                        'activity_key' : 'concept:name',
                                                        'resource_key' : 'org:resource_start',
                                                        'timestamp_key' : 'time:timestamp_start',
                                                        'categorical_args' : ['concept_name',
                                                                              'day_of_week',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda inter_instance_counts, inter_instance_column_names : [0 if inter_instance_column_name not in inter_instance_counts else inter_instance_counts[inter_instance_column_name] for inter_instance_column_name in inter_instance_column_names])(inter_instance_counts, self.inter_instance_column_names)'
                                                                             ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'inter_instance_column_names' : ii3,
                                                        'strict_parsing' : False,
                                                    },
                                     test_event_log, n_processes=n_processes, batch_size=batch_size
                                )
likelihoods_A = evaluator_A.sample_cases(False, multiprocessing=True)

100%|██████████| 1202/1202 [00:06<00:00, 190.53it/s]


In [7]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-Infinity')

In [8]:
np.mean(get_pscores(likelihoods_A))

np.float64(15669.431483996235)